**Import Libraries and Load Dataset**

Dataset contains mixed types and placeholder missing values (?).

Larger and more categorical-heavy.

In [22]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

In [23]:
adult_data = pd.read_csv('drive/MyDrive/Datasets For ML/Adult_Income_Dataset.csv')

print("Dataset Shape:", adult_data.shape)
print("\nColumn Names:", adult_data.columns)
print("\nData Types:")
adult_data.dtypes

Dataset Shape: (48842, 15)

Column Names: Index(['age', 'workclass', 'fnlwgt', 'education', 'educational-num',
       'marital-status', 'occupation', 'relationship', 'race', 'gender',
       'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
       'income'],
      dtype='object')

Data Types:


,0
age,int64
workclass,object
fnlwgt,int64
education,object
educational-num,int64
marital-status,object
occupation,object
relationship,object
race,object
gender,object


In [24]:
adult_data.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


**Replace Placeholder Missing Values**

Adult dataset uses ? instead of NaN.

Must be normalized before imputation.

In [25]:
adult_data.replace('?', np.nan, inplace=True)

adult_data.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K


**Drop Low-Value or Redundant Columns**

education duplicates education-num.

fnlwgt not useful for prediction.

In [26]:
adult_data.drop(columns=['fnlwgt', 'education'], inplace=True)
print("Shape after dropping fnlwgt and education:", adult_data.shape)

Shape after dropping fnlwgt and education: (48842, 13)


**Separate Numeric and Categorical Columns**

Required for KNN Imputer and encoding pipeline.

In [27]:
num_cols = adult_data.select_dtypes(include=['int64', 'float64']).columns
cat_cols = adult_data.select_dtypes(include=['object']).columns

print("Numeric Columns:", num_cols)
print("\nCategorical Columns:", cat_cols)

Numeric Columns: Index(['age', 'educational-num', 'capital-gain', 'capital-loss',
       'hours-per-week'],
      dtype='object')

Categorical Columns: Index(['workclass', 'marital-status', 'occupation', 'relationship', 'race',
       'gender', 'native-country', 'income'],
      dtype='object')


**Handle Missing Values (Categorical Only)**

Numeric features have no missing values, so no imputation is required.

In [28]:
adult_data1 = adult_data.copy()
adult_data1.isna().sum()

,0
age,0
workclass,2799
educational-num,0
marital-status,0
occupation,2809
relationship,0
race,0
gender,0
capital-gain,0
capital-loss,0


In [29]:
for col in cat_cols:
    adult_data1[col].fillna(adult_data1[col].mode()[0], inplace=True)

print("Missing values after categorical handling:\n")
adult_data1.isna().sum()

Missing values after categorical handling:



/tmp/ipykernel_11455/2318237356.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  adult_data1[col].fillna(adult_data1[col].mode()[0], inplace=True)


,0
age,0
workclass,0
educational-num,0
marital-status,0
occupation,0
relationship,0
race,0
gender,0
capital-gain,0
capital-loss,0


**Handle Missing Values Using SimpleImputer (Categorical Only)**

In [30]:
adult_data2 = adult_data.copy()
adult_data2.isna().sum()

,0
age,0
workclass,2799
educational-num,0
marital-status,0
occupation,2809
relationship,0
race,0
gender,0
capital-gain,0
capital-loss,0


In [31]:
# Initialize imputer for categorical data
cat_imputer = SimpleImputer(strategy='most_frequent')

# Apply imputer
adult_data2[cat_cols] = cat_imputer.fit_transform(adult_data2[cat_cols])

# Verify missing values
print("Missing values after SimpleImputer:")
adult_data2.isna().sum()

Missing values after SimpleImputer:


,0
age,0
workclass,0
educational-num,0
marital-status,0
occupation,0
relationship,0
race,0
gender,0
capital-gain,0
capital-loss,0


**Target Encoding**

Binary target.

Required before stratified split.

In [32]:
adult_data['income'].value_counts()

,count
income,
<=50K,37155
>50K,11687


In [33]:
adult_data['income'] = adult_data['income'].map({'<=50K': 0, '>50K': 1})

print("Income distribution:")
adult_data['income'].value_counts()

Income distribution:


,count
income,
0,37155
1,11687


**Large-Scale OneHotEncoding (Categorical Features)**

Adult dataset has many nominal categories. OneHotEncoding is required to avoid ordinal bias.

In [34]:
adult_data.shape

(48842, 13)

In [35]:
adult_data = pd.get_dummies(
    adult_data,
    columns=cat_cols.drop('income'),
    drop_first=True
)

print("Shape after OneHotEncoding:", adult_data.shape)

Shape after OneHotEncoding: (48842, 82)


**Feature Scaling (StandardScaler)**

StandardScaler is preferred due to wide numeric ranges and linear model assumptions.

In [36]:
adult_data[num_cols].head(3)

,age,educational-num,capital-gain,capital-loss,hours-per-week
0,25,7,0,0,40
1,38,9,0,0,50
2,28,12,0,0,40


In [37]:
scaler = StandardScaler()
adult_data[num_cols] = scaler.fit_transform(adult_data[num_cols])

In [38]:
adult_data[num_cols].head(3)

,age,educational-num,capital-gain,capital-loss,hours-per-week
0,-0.995129,-1.197259,-0.144804,-0.217127,-0.034087
1,-0.046942,-0.419335,-0.144804,-0.217127,0.772930
2,-0.776316,0.747550,-0.144804,-0.217127,-0.034087


**Correlation-Based Feature Inspection**

Additional analytical step not used in Titanic.

In [39]:
adult_data['income'].head()

,income
0,0
1,0
2,1
3,1
4,0


In [40]:
corr = adult_data.corr()['income'].abs().sort_values(ascending=False)

print("Top correlated features with income:\n")
corr.head()

Top correlated features with income:



,income
income,1.000000
marital-status_Married-civ-spouse,0.445853
educational-num,0.332613
marital-status_Never-married,0.318782
age,0.230369


**Feature–Target Split**

In [41]:
X = adult_data.drop(columns=['income'])
Y = adult_data[['income']]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: (48842, 81)
Y shape: (48842, 1)


**Train–Test Split with Strong Stratification**

Class imbalance is more severe than Titanic. Stratification is mandatory.

In [42]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    train_size=0.7,
    stratify=Y,
    random_state=3
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain target distribution:\n", Y_train.value_counts(normalize=True))
print("\nTest target distribution:\n", Y_test.value_counts(normalize=True))

Train shape: (34189, 81)
Test shape: (14653, 81)

Train target distribution:
 income
0         0.760713
1         0.239287
Name: proportion, dtype: float64

Test target distribution:
 income
0         0.760732
1         0.239268
Name: proportion, dtype: float64


**Conclusion**

The Adult dataset was cleaned, encoded, and scaled to handle missing values and categorical features. A stratified train-test split ensures balanced class distribution for reliable model training and evaluation.
